In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 'hasha' 디렉터리의 내용 확인하기
!ls "/content/drive/My Drive/Hasha"

# Python 코드에서 경로 변수 설정하기
import os
hasha_dir = "/content/drive/My Drive/Hasha"

# 해당 디렉터리가 존재하는지 확인
if os.path.exists(hasha_dir):
  print(f"'{hasha_dir}' 디렉터리를 찾았습니다.")
else:
  print(f"경로를 찾을 수 없습니다. 'Hasha' 디렉터리의 위치를 확인해주세요.")

BTC_data_all.csv  data	    lstm_reinforcement.ipynb
BTC_data.csv	  data.csv  ppo_model
'/content/drive/My Drive/Hasha' 디렉터리를 찾았습니다.


In [21]:
!pip install pandas_ta
!pip install stable_baselines3
!pip install vectorbt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.7/527.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.7 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import pandas_ta as ta

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 데이터

In [5]:
import pandas as pd
import pandas_ta as ta

# 1. 4시간봉 데이터 로드
price_data_4h = pd.read_csv(f"{hasha_dir}/data/BTC_4h_data_all.csv", encoding="utf-8-sig", index_col=0, parse_dates=True)

# 2. 일봉으로 리샘플링
ohlc_dict = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}
price_data_1d = price_data_4h.resample('D').apply(ohlc_dict)
price_data_1d.dropna(inplace=True)

# --- 4시간봉 피처 생성 ---
features_4h = price_data_4h.copy()
features_4h.ta.rsi(length=14, append=True, col_names=('RSI_14_4H',))
features_4h.ta.macd(fast=12, slow=26, signal=9, append=True, col_names=('MACD_12_26_9_4H', 'MACDh_12_26_9_4H', 'MACDs_12_26_9_4H'))
features_4h.ta.bbands(length=20, std=2, append=True, col_names=('BBL_20_2.0_4H', 'BBM_20_2.0_4H', 'BBU_20_2.0_4H', 'BBB_20_2.0_4H', 'BBP_20_2.0_4H'))
# 원본 가격 데이터는 나중에 합칠 것이므로 피처셋에서는 제거
features_4h.drop(['Open', 'High', 'Low', 'Close', 'Volume'], axis=1, inplace=True)


# --- 일봉 피처 생성 ---
features_1d = price_data_1d.copy()
features_1d.ta.rsi(length=14, append=True, col_names=('RSI_14_1D',))
features_1d.ta.sma(length=50, append=True, col_names=('SMA_50_1D',))
features_1d.ta.adx(length=14, append=True, col_names=('ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D'))
# 원본 가격 데이터 제거
features_1d.drop(['Open', 'High', 'Low', 'Close', 'Volume'], axis=1, inplace=True)


# --- 피처 결합 ---

# 일봉 피처를 4시간봉 인덱스에 맞게 재정렬 (ffill)
aligned_features_1d = features_1d.reindex(features_4h.index, method='ffill')

# (원본 4H 가격) + (4H 피처) + (정렬된 1D 피처) 결합
final_features = pd.concat([price_data_4h, features_4h, aligned_features_1d], axis=1)

# 피처 계산 초기의 NaN 값 제거
final_features.dropna(inplace=True)

print(final_features.columns)
final_features.head()

[!] Not enough col_names were specified : got 3, expected 4.
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14_4H',
       'MACD_12_26_9_4H', 'MACDh_12_26_9_4H', 'MACDs_12_26_9_4H',
       'BBL_20_2.0_4H', 'BBM_20_2.0_4H', 'BBU_20_2.0_4H', 'BBB_20_2.0_4H',
       'BBP_20_2.0_4H', 'RSI_14_1D', 'SMA_50_1D'],
      dtype='object')


,Open,High,Low,Close,Volume,RSI_14_4H,MACD_12_26_9_4H,MACDh_12_26_9_4H,MACDs_12_26_9_4H,BBL_20_2.0_4H,BBM_20_2.0_4H,BBU_20_2.0_4H,BBB_20_2.0_4H,BBP_20_2.0_4H,RSI_14_1D,SMA_50_1D
Datetime,,,,,,,,,,,,,,,,
2017-11-13 00:00:00,7349000.0,7649000.0,7141000.0,7297000.0,1.738097e+09,43.131008,-242868.273287,-16272.417052,-226595.856235,6.774005e+06,7615400.0,8.456795e+06,22.097209,0.310791,53.985687,6376620.0
2017-11-13 04:00:00,7300000.0,7344000.0,6845000.0,6860000.0,1.669486e+09,36.923241,-267891.895200,-33036.831172,-234855.064028,6.684079e+06,7551200.0,8.418321e+06,22.966455,0.101440,53.985687,6376620.0
2017-11-13 08:00:00,6878000.0,7361000.0,6850000.0,7104000.0,1.699718e+09,41.947370,-264980.010429,-24099.957120,-240880.053308,6.652196e+06,7501700.0,8.351204e+06,22.648317,0.265922,53.985687,6376620.0
2017-11-13 12:00:00,7139000.0,7670000.0,7106000.0,7400000.0,1.755850e+09,47.418887,-236066.368395,3850.947931,-239917.316326,6.664746e+06,7466650.0,8.268554e+06,21.479616,0.458443,53.985687,6376620.0
2017-11-13 16:00:00,7410000.0,7470000.0,7183000.0,7470000.0,1.759918e+09,48.651439,-205138.969441,27822.677508,-232961.646949,6.695595e+06,7433400.0,8.171205e+06,19.851088,0.524803,53.985687,6376620.0


# 전처리

In [7]:
from sklearn.preprocessing import MinMaxScaler
import joblib, os

# 시간 순서에 따른 데이터 분할
train_data = final_features.loc[:'2024-06']
validation_data = final_features.loc['2024-07':'2024-12']
test_data = final_features.loc['2025':]

# 스케일러 훈련 및 적용
scaler = MinMaxScaler()
scaled_train_features = scaler.fit_transform(train_data)
scaled_validation_features = scaler.transform(validation_data)
scaled_test_features = scaler.transform(test_data)

save_dir = f"{hasha_dir}/ppo_models/"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(scaler, f"{hasha_dir}/ppo_models/scaler_ver1.pkl")



['/content/drive/My Drive/Hasha/ppo_models/scaler_ver1.pkl']

# 환경 구축

In [12]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class TradingEnv(gym.Env):
    """
    Action Space: Discrete(2)
        - 0: Sell (매도 또는 현금 보유 유지)
        - 1: Buy (매수 또는 포지션 보유 유지)
    """

    def __init__(self,
                 scaled_features_array,
                 original_close_prices,
                 lookback_window,
                 initial_balance=1000000,
                 fee=0.0005,
                 slippage=0.0005):

        super(TradingEnv, self).__init__()

        # 데이터
        self.features = scaled_features_array # 정규화된 피처 데이터 (관측용)
        self.prices = original_close_prices   # 원본 종가 데이터 (계산용)
        self.lookback_window = lookback_window
        self.num_features = scaled_features_array.shape[1]
        self.total_steps = len(self.prices)

        # Vectorbt와 동일한 환경 설정
        self.initial_balance = initial_balance
        self.fee = fee
        self.slippage = slippage

        # Action Space (0: Sell, 1: Buy)
        self.action_space = spaces.Discrete(2)

        # Observation Space (MinMaxScaler로 0~1 정규화됨)
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(lookback_window, self.num_features),
            dtype=np.float32
        )

        # 포트폴리오 상태 변수
        self.balance = 0
        self.btc_held = 0
        self.position = 0 # 0: 현금 보유 (No position), 1: BTC 보유 (Long position)
        self.current_step = 0

    def _get_observation(self):
        """현재 스텝의 관측값(State)을 반환 (정규화된 피처)"""
        return self.features[self.current_step - self.lookback_window : self.current_step]

    def _get_portfolio_value(self, price):
        """현재 포트폴리오의 총 가치 반환"""
        return self.balance + (self.btc_held * price)

    def reset(self, seed=None):
        super().reset(seed=seed)

        self.balance = self.initial_balance
        self.btc_held = 0
        self.position = 0 # 0: 현금
        self.current_step = self.lookback_window

        observation = self._get_observation()
        info = {}

        return observation, info

    def step(self, action):
        # 1. 직전 스텝의 정보 저장
        current_price = self.prices[self.current_step]
        prev_portfolio_value = self._get_portfolio_value(current_price)

        # 2. 행동(Action) 수행 (핵심 로직)
        if action == 1: # 1: Buy Signal
            if self.position == 0: # 현금 보유 상태 -> 매수 실행
                buy_price = current_price * (1 + self.slippage)
                cost = buy_price * (1 + self.fee)
                self.btc_held = self.balance / cost
                self.balance = 0
                self.position = 1 # 포지션 'Long'으로 변경

            # (else: self.position == 1: # 이미 포지션 보유 -> 암묵적 'Hold')

        elif action == 0: # 0: Sell Signal
            if self.position == 1: # 포지션 보유 상태 -> 매도 실행
                sell_price = current_price * (1 - self.slippage)
                revenue = self.btc_held * sell_price
                self.balance = revenue * (1 - self.fee)
                self.btc_held = 0
                self.position = 0 # 포지션 'Cash'로 변경

            # (else: self.position == 0: # 이미 현금 보유 -> 암묵적 'Hold')

        # 3. 시간 이동
        self.current_step += 1

        # 4. 종료(Done) 여부 확인
        done = self.current_step >= self.total_steps - 1

        # 5. 보상(Reward) 계산
        next_price = self.prices[self.current_step] if not done else current_price
        current_portfolio_value = self._get_portfolio_value(next_price)

        # 보상 = 포트폴리오 가치 변화량
        reward = current_portfolio_value - prev_portfolio_value

        # 6. 다음 상태(Observation) 및 정보 반환
        observation = self._get_observation() if not done else self.observation_space.sample()
        truncated = False # (우선 False로 설정)
        info = {'portfolio_value': current_portfolio_value} # (로그용)

        return observation, reward, done, truncated, info

# 아키텍처

In [13]:
import torch as th
from torch import nn
from gymnasium import spaces
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class CustomCnnLstmExtractor(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the Prioritized Replay Buffer.
    """
    def __init__(self, observation_space: spaces.Box, features_dim: int = 128):
        super().__init__(observation_space, features_dim)

        # observation_space.shape[1] = num_features (예: 18)
        # observation_space.shape[0] = lookback_window (예: 30)
        n_input_features = observation_space.shape[1]
        lookback = observation_space.shape[0]

        # 1. 1D-CNN Layer: 단기 패턴 추출
        self.cnn = nn.Sequential(
            # (Batch, Features, Lookback)
            nn.Conv1d(in_channels=n_input_features, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2),
            nn.ReLU(),
            # (Batch, 128, Lookback)
        )

        # CNN을 통과한 뒤의 LSTM 입력 차원
        self.lstm = nn.LSTM(input_size=128, hidden_size=features_dim)

    def forward(self, observations: th.Tensor) -> th.Tensor:
        # Pytorch Conv1D는 (Batch, Channels=Features, Length=Lookback) 형태를 기대합니다.
        # SB3에서 오는 observations는 (Batch, Lookback, Features) 이므로 차원 변경
        observations = observations.permute(0, 2, 1)

        # (Batch, Features, Lookback) -> (Batch, 128, Lookback)
        cnn_features = self.cnn(observations)

        # Pytorch LSTM은 (Length=Lookback, Batch, Features) 형태를 기대합니다.
        # (Batch, 128, Lookback) -> (Lookback, Batch, 128)
        lstm_input = cnn_features.permute(2, 0, 1)

        # LSTM의 마지막 hidden_state를 특징으로 사용
        # (h_n, c_n)
        _, (hidden_state, _) = self.lstm(lstm_input)

        # (Num_layers, Batch, features_dim) -> (Batch, features_dim)
        # Squeeze(0)으로 맨 앞의 1차원(num_layers)을 제거
        return hidden_state.squeeze(0)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
from stable_baselines3.common.monitor import Monitor
# (TradingEnv 클래스 정의는 여기에 있다고 가정)

# --- 1. 훈련(Train) 환경 생성 ---
train_close_prices = train_data['Close'].values
train_env = TradingEnv(
    scaled_features_array=scaled_train_features,
    original_close_prices=train_close_prices,
    lookback_window=30, # lookback
    initial_balance=1000000,
    fee=0.0005,
    slippage=0.0005
)
train_env = Monitor(train_env, f"{hasha_dir}/ppo_log/train_monitor/")

# --- 2. 검증(Validation) 환경 생성 ---
validation_close_prices = validation_data['Close'].values
validation_env = TradingEnv(
    scaled_features_array=scaled_validation_features,
    original_close_prices=validation_close_prices,
    lookback_window=30,
    initial_balance=1000000,
    fee=0.0005,
    slippage=0.0005
)
validation_env = Monitor(validation_env,  f"{hasha_dir}/ppo_log/val_monitor/")

In [15]:
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
import os

class SaveOnBestCallback(BaseCallback):
    """
    Validation 환경에서 최고 보상(reward)을 달성할 때마다 모델을 저장하는 콜백
    """
    def __init__(self, check_freq: int, eval_env: gym.Env, save_path: str, verbose=1):
        super(SaveOnBestCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.eval_env = eval_env
        self.save_path = save_path
        self.best_mean_reward = -np.inf

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            # 현재 모델로 validation 환경에서 평가
            mean_reward, _ = evaluate_policy(self.model, self.eval_env, n_eval_episodes=1)

            if self.verbose > 0:
                print(f"Step {self.n_calls}: Validation Mean Reward: {mean_reward:.2f}")

            # 최고 보상 갱신 시 모델 저장
            if mean_reward > self.best_mean_reward:
                self.best_mean_reward = mean_reward
                self.model.save(os.path.join(self.save_path, "best_model"))
                if self.verbose > 0:
                    print(f"New best model saved! Mean Reward: {mean_reward:.2f}")
        return True

In [16]:
from stable_baselines3 import PPO

# 1. 저장 경로 생성
save_dir =  f"{hasha_dir}/ppo_models/"
log_dir =  f"{hasha_dir}/ppo_tensorboard_log/"
os.makedirs(save_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# 2. 커스텀 '뇌' 사용 설정
policy_kwargs = dict(
    features_extractor_class=CustomCnnLstmExtractor,
    features_extractor_kwargs=dict(features_dim=128), # LSTM의 hidden_size
)

# 3. 콜백 초기화
# 2000 스텝마다 validation 환경에서 성능을 체크
callback = SaveOnBestCallback(check_freq=2000, eval_env=validation_env, save_path=save_dir)

# 4. PPO 모델 초기화
model = PPO(
    "MlpPolicy", # "MlpPolicy"는 Actor/Critic 헤드 구조를 의미. '뇌'는 policy_kwargs로 대체됨
    train_env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    n_steps=1024, # (조정 가능한 하이퍼파라미터)
    batch_size=64,
    n_epochs=10,
    tensorboard_log=log_dir
)

# 5. 모델 학습 시작!
print("--- 모델 학습 시작 ---")
model.learn(
    total_timesteps=500_000,  # (총 학습 스텝, 충분히 길게 설정)
    callback=callback
)
print("--- 모델 학습 완료 ---")

# 6. 최종 모델 저장 (콜백이 "best_model"을 저장했지만, 마지막 모델도 저장)
model.save(os.path.join(save_dir, "final_model"))

Using cuda device
Wrapping the env in a DummyVecEnv.
--- 모델 학습 시작 ---
Logging to /content/drive/My Drive/Hasha/ppo_tensorboard_log/PPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
|    entropy_loss         | -0.008       |
|    explained_variance   | 0            |
|    learning_rate        | 0.0003       |
|    loss                 | 6.97e+03     |
|    n_updates            | 2560         |
|    policy_gradient_loss | -0.000334    |
|    value_loss           | 6.14e+04     |
------------------------------------------
Step 264000: Validation Mean Reward: 0.00
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.45e+04      |
|    ep_rew_mean          | -6.62e+05     |
| time/                   |               |
|    fps                  | 267           |
|    iterations           | 258           |
|    time_elapsed         | 987           |
|    total_timesteps      | 264192        |
| train/                  |               |
|    approx_kl            | 3.5390258e-08 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    e

# 테스트

In [18]:
import pandas as pd
import numpy as np
import joblib
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# (TradingEnv, CustomCnnLstmExtractor 클래스 정의는 여기에 있다고 가정)

# --- 1. 테스트 데이터 및 스케일러 준비 ---
# 이 변수들은 이전 단계에서 준비되었다고 가정합니다.
lookback = 30

# 스케일러 로드
scaler_path = f'{hasha_dir}/ppo_models/scaler_ver1.pkl'
scaler = joblib.load(scaler_path)

# test_data 스케일링 (fit_transform이 아닌 transform 사용)
scaled_test_features = scaler.transform(test_data)
test_close_prices = test_data['Close'].values

# --- 2. 테스트 환경 생성 ---
test_env = TradingEnv(
    scaled_features_array=scaled_test_features,
    original_close_prices=test_close_prices,
    lookback_window=lookback,
    initial_balance=1000000,
    fee=0.0005,
    slippage=0.0005
)

# --- 3. 훈련된 '최고의' 모델 로드 ---
model_path = f"{hasha_dir}/ppo_models/best_model.zip"
model = PPO.load(model_path, env=test_env)

print("테스트 환경 및 모델 로드 완료.")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
테스트 환경 및 모델 로드 완료.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
### 2단계: 백테스트 실행 (시그널 생성) - (수정된 코드) ###

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning) # Numpy 경고 무시

obs, info = test_env.reset()
done = False
truncated = False

actions_list = []
portfolio_values = []

# 총 스텝 수에서 lookback 윈도우만큼 뺀 횟수 (정확히 1344번) 만큼 반복합니다.
num_steps_to_predict = test_env.total_steps - test_env.lookback_window

print(f"백테스트 시뮬레이션 시작 (총 {num_steps_to_predict} 스텝)...")

for _ in range(num_steps_to_predict):
    # deterministic=True로 설정하여 평가 (가장 확률 높은 행동 선택)
    action, _states = model.predict(obs, deterministic=True)

    obs, reward, done, truncated, info = test_env.step(action)

    actions_list.append(int(action))
    portfolio_values.append(info.get('portfolio_value', test_env.balance)) # 정보 저장

    # # (이론상 for 루프 횟수와 done 시점이 일치해야 하지만, 안전장치로 남겨둡니다)
    # if done or truncated:
    #     break

print("백테스트 시뮬레이션 완료.")
# 이제 actions_list의 길이는 1344가 됩니다.

백테스트 시뮬레이션 시작 (총 1344 스텝)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


백테스트 시뮬레이션 완료.


In [31]:
actions_list

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [27]:
import pandas as pd

# 1. 테스트 데이터의 실제 종가 (vectorbt 입력용)
price_close_test = test_data['Close']
if price_close_test.index.tz is None:
    price_close_test.index = price_close_test.index.tz_localize('UTC')

# 2. Action 리스트를 Pandas Series로 변환
# actions_list는 lookback 시점부터 시작했으므로, 해당 인덱스에 맞춤
signal_index = price_close_test.index[lookback:]
signals_raw = pd.Series(actions_list, index=signal_index)

# 3. 전체 기간(price_close_test)에 맞게 리인덱싱 (앞부분은 0(현금)으로 채움)
signals = signals_raw.reindex(price_close_test.index, fill_value=0)

# 4. 'entries'와 'exits' 생성
# entries: 0(현금) -> 1(보유)로 바뀐 시점
entries = (signals == 1) & (signals.shift(1) == 0)

# exits: 1(보유) -> 0(현금)으로 바뀐 시점
exits = (signals == 0) & (signals.shift(1) == 1)

# vectorbt가 True/False 값을 선호하므로 boolean으로 유지

In [29]:
import vectorbt as vbt

# vectorbt 설정
vbt.settings.set_theme('dark')
vbt.settings.plotting['layout']['width'] = 1000
vbt.settings.plotting['layout']['height'] = 600

# Portfolio 생성
portfolio = vbt.Portfolio.from_signals(
    close=price_close_test,
    entries=entries,
    exits=exits,
    init_cash=1000000,
    fees=0.0005,    # 0.05%
    slippage=0.0005, # 0.05%
    freq='4h',      # 데이터 빈도 명시
)

# --- 최종 성과 리포트 ---
print("\n--- 강화학습 모델 백테스트 성과 (Test Data) ---")
print(portfolio.stats())

# --- 수익률 그래프 시각화 ---
print("\n수익률 그래프를 플로팅합니다...")
fig = portfolio.plot(subplots=['cum_returns', 'orders', 'drawdowns'])
fig.show()

# --- (참고) Buy & Hold (존버) 성과와 비교 ---
bnh_returns = vbt.Portfolio.from_holding(price_close_test, init_cash=1000000).total_return()

print(f"\n* 참고: 동일 기간 Buy & Hold 수익률: {bnh_returns * 100:.2f}%")


--- 강화학습 모델 백테스트 성과 (Test Data) ---
Start                         2025-01-01 00:00:00+00:00
End                           2025-08-17 20:00:00+00:00
Period                                229 days 00:00:00
Start Value                                   1000000.0
End Value                                1131731.097334
Total Return [%]                               13.17311
Benchmark Return [%]                          16.240465
Max Gross Exposure [%]                            100.0
Total Fees Paid                              499.750125
Max Drawdown [%]                              30.518257
Max Drawdown Duration                 174 days 16:00:00
Total Trades                                          1
Total Closed Trades                                   0
Total Open Trades                                     1
Open Trade PnL                            131731.097334
Win Rate [%]                                        NaN
Best Trade [%]                                      NaN
Worst Trade

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).




* 참고: 동일 기간 Buy & Hold 수익률: 16.24%
